# AI Hub 한국인 피부상태 측정 데이터 EDA

**목표**
1. 라벨 JSON 구조 파악
2. 어노테이션(전문의 평가) 분포 확인
3. 장비 측정값 분포 확인
4. 메타데이터 분포 (연령/성별/피부타입)
5. 결측치 및 클래스 불균형 파악
6. **학습 타겟 라벨 결정**

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 1. 경로 설정

In [ ]:
DATA_ROOT = Path("../028.한국인 피부상태 측정 데이터/3.개방데이터/1.데이터")

META_CSV     = DATA_ROOT / "Other/extracted/메타데이터/meta_data.csv"
MEASURE_CSV  = DATA_ROOT / "Other/extracted/메타데이터/measurement_data.csv"

VL_LABEL_DIR = DATA_ROOT / "Validation/02.라벨링데이터/extracted"
TL_LABEL_DIR = DATA_ROOT / "Training/02.라벨링데이터/extracted"   # TL.zip 압축 해제 후 사용

# VS.zip / TS.zip은 이미지 (현재 압축 해제 보류)
VL_IMG_DIR   = DATA_ROOT / "Validation/01.원천데이터/extracted"
TL_IMG_DIR   = DATA_ROOT / "Training/01.원천데이터/extracted"

print("메타데이터 존재:", META_CSV.exists())
print("측정데이터 존재:", MEASURE_CSV.exists())
print("VL 라벨 디렉토리 존재:", VL_LABEL_DIR.exists())

## 2. 메타데이터 (meta_data.csv)

In [ ]:
meta_df = pd.read_csv(META_CSV, encoding='utf-8')
print(f"피험자 수: {len(meta_df)}")
print(f"컬럼: {list(meta_df.columns)}")
meta_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 성별 분포
meta_df['성별'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF9999','#9999FF'])
axes[0].set_title('성별 분포'); axes[0].set_xlabel('')

# 연령 분포
axes[1].hist(meta_df['나이'], bins=20, edgecolor='black', color='#66B2FF')
axes[1].set_title('연령 분포'); axes[1].set_xlabel('나이')

# 피부타입 분포
meta_df['얼굴피부타입'].value_counts().plot(kind='barh', ax=axes[2], color='#99FF99')
axes[2].set_title('피부타입 분포')

# 민감도 분포
meta_df['자가민감여부'].value_counts().plot(kind='bar', ax=axes[3], color='#FFCC66')
axes[3].set_title('자가민감 여부'); axes[3].set_xlabel('')

plt.tight_layout()
plt.show()

print("\n피부타입별 빈도:")
print(meta_df['얼굴피부타입'].value_counts())
print("\n연령 통계:")
print(meta_df['나이'].describe())

## 3. 장비 측정 데이터 (measurement_data.csv)

In [ ]:
meas_df = pd.read_csv(MEASURE_CSV, encoding='utf-8')
print(f"행 수: {len(meas_df)}, 컬럼 수: {len(meas_df.columns)}")
print("\n컬럼 목록:")
for i, col in enumerate(meas_df.columns):
    print(f"  [{i:3d}] {col}")

In [ ]:
# 핵심 측정값만 추출: 수분(4개 부위), 스팟, 모공
moisture_cols = [c for c in meas_df.columns if c.startswith('수분_')]
pore_cols     = [c for c in meas_df.columns if c.startswith('모공개수_')]
spot_col      = '스팟개수_정면'

print("수분 컬럼:", moisture_cols)
print("모공 컬럼:", pore_cols)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(moisture_cols):
    ax = axes[0][i]
    ax.hist(meas_df[col].dropna(), bins=30, edgecolor='black', color='#66B2FF')
    ax.set_title(col); ax.set_xlabel('수분량')

for i, col in enumerate(pore_cols):
    ax = axes[1][i]
    ax.hist(meas_df[col].dropna(), bins=30, edgecolor='black', color='#99FF99')
    ax.set_title(col)

ax = axes[1][2]
ax.hist(meas_df[spot_col].dropna(), bins=30, edgecolor='black', color='#FFCC66')
ax.set_title(spot_col)

plt.tight_layout()
plt.show()

In [ ]:
# meta + measurement 병합 확인
merged = meta_df.merge(meas_df, on='subject_no', how='left')
print(f"병합 결과: {len(merged)}행")
print(f"측정값 결측률:")
print(merged[moisture_cols + pore_cols + [spot_col]].isnull().mean().round(3))

## 4. JSON 라벨 파싱 (Validation Set)

### 4.1 데이터 구조 요약 (실제 확인값)
```
파일명: {subject_id}_{session}_{view}_{facepart}.json
뷰(view): F, Fb, Ft, L15, L30, R15, R30  (7가지 각도)
부위(facepart): 0~8  (9개 얼굴 부위)
장비: 0=디지털카메라 / 1=스마트패드 / 2=스마트폰

skin_type 코드: 0=건성, 1=지성, 3=복합건성, 4=복합지성, 5=중성

facepart별 annotation (실제 측정 스케일):
  0: acne                 → 병변 좌표 리스트 [{name, points}, ...]
                            null=여드름 없음 / list=병변 위치+유형
                            name 종류: papule, pustule, cyst, comedone
  1: forehead_pigmentation  (이마 색소침착, 0-5)
     forehead_wrinkle       (이마 주름,     0-6)  ★ 베이스라인 추천
  2: glabellus_wrinkle      (미간 주름,     0-6)
  3: l_perocular_wrinkle    (왼눈가 주름,   0-6)
  4: r_perocular_wrinkle    (오른눈가 주름, 0-6)
  5: l_cheek_pore           (왼볼 모공,     0-4)
     l_cheek_pigmentation   (왼볼 색소침착, 0-5)
  6: r_cheek_pore           (오른볼 모공,   0-4)
     r_cheek_pigmentation   (오른볼 색소침착,0-5)
  7: lip_dryness            (입술 건조도,   0-4)
  8: chin_sagging           (턱선처짐,      0-5)
```

In [ ]:
def parse_label_dir(label_dir: Path) -> pd.DataFrame:
    """라벨 디렉토리의 모든 JSON을 파싱하여 DataFrame 반환."""
    records = []
    for json_file in label_dir.rglob("*.json"):
        with open(json_file, encoding='utf-8') as f:
            try:
                d = json.load(f)
            except json.JSONDecodeError:
                continue

        info   = d.get('info', {})
        images = d.get('images', {})
        annot  = d.get('annotations', {}) or {}
        equip  = d.get('equipment', {}) or {}

        record = {
            'json_file':  json_file.name,
            'subject_id': info.get('id'),
            'filename':   info.get('filename'),
            'gender':     info.get('gender'),
            'age':        info.get('age'),
            'skin_type':  info.get('skin_type'),
            'sensitive':  info.get('sensitive'),
            'device':     images.get('device'),
            'angle':      images.get('angle'),
            'facepart':   images.get('facepart'),
            'width':      images.get('width'),
            'height':     images.get('height'),
        }
        record.update({f'ann_{k}': v for k, v in annot.items()})
        record.update({f'eq_{k}':  v for k, v in equip.items()})
        records.append(record)

    return pd.DataFrame(records)


print("Validation 라벨 파싱 중...")
vl_df = parse_label_dir(VL_LABEL_DIR)
print(f"총 {len(vl_df)}개 레코드, {len(vl_df.columns)}개 컬럼")
vl_df.head(3)

In [ ]:
# 어노테이션 컬럼 목록 확인
ann_cols = [c for c in vl_df.columns if c.startswith('ann_')]
eq_cols  = [c for c in vl_df.columns if c.startswith('eq_')]
print(f"어노테이션 컬럼 ({len(ann_cols)}개):")
for c in ann_cols: print(f"  {c}")
print(f"\n장비측정 컬럼 ({len(eq_cols)}개):")
for c in eq_cols[:15]: print(f"  {c}")
if len(eq_cols) > 15: print(f"  ... 외 {len(eq_cols)-15}개")

## 5. 어노테이션 분포 (핵심: 학습 타겟)

In [ ]:
# facepart별로 어노테이션 non-null 수 확인
# ※ acne는 object detection 형식(리스트)이므로 별도 처리
print("어노테이션별 유효 샘플 수 및 분포:")
print(f"{'컬럼명':<35} {'non-null':>8} {'결측률':>8}  분포")
print('-' * 80)
for col in ann_cols:
    valid_mask = vl_df[col].notna()
    null_rate = 1 - valid_mask.mean()
    valid_count = valid_mask.sum()
    if valid_count == 0:
        continue
    sample_vals = vl_df.loc[valid_mask, col].iloc[0]
    if isinstance(sample_vals, list):
        # acne: 병변 좌표 리스트
        lesion_counts = vl_df.loc[valid_mask, col].apply(lambda x: len(x) if isinstance(x, list) else 0)
        print(f"{col:<35} {valid_count:>8} {null_rate:>8.1%}  [OD형식] 병변수 min={lesion_counts.min()} max={lesion_counts.max()} mean={lesion_counts.mean():.1f}")
    else:
        try:
            dist = dict(vl_df.loc[valid_mask, col].astype(float).astype(int).value_counts().sort_index())
            print(f"{col:<35} {valid_count:>8} {null_rate:>8.1%}  {dist}")
        except Exception:
            print(f"{col:<35} {valid_count:>8} {null_rate:>8.1%}  [파싱오류]")

In [ ]:
# 핵심 어노테이션 시각화 (acne 제외 - OD 형식)
numeric_anns = [
    'ann_forehead_pigmentation', 'ann_forehead_wrinkle',
    'ann_glabellus_wrinkle',
    'ann_l_perocular_wrinkle',   'ann_r_perocular_wrinkle',
    'ann_l_cheek_pore',          'ann_l_cheek_pigmentation',
    'ann_lip_dryness',           'ann_chin_sagging'
]
numeric_anns = [c for c in numeric_anns if c in vl_df.columns]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_anns):
    ax = axes[i]
    data = vl_df[col].dropna().astype(float).astype(int)
    counts = data.value_counts().sort_index()
    bars = ax.bar(counts.index.astype(str), counts.values, color='steelblue', edgecolor='black')
    ax.set_title(col.replace('ann_', ''), fontsize=10)
    ax.set_xlabel('등급')
    ax.set_ylabel('빈도')
    total = len(data)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
                f'{v/total:.0%}', ha='center', fontsize=8)

for j in range(len(numeric_anns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('전문의 어노테이션 분포 (Validation Set, acne 제외)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# acne 별도 시각화 (병변 수 기반)
if 'ann_acne' in vl_df.columns:
    acne_valid = vl_df[vl_df['ann_acne'].notna()]['ann_acne']
    acne_counts = acne_valid.apply(lambda x: len(x) if isinstance(x, list) else 0)
    total_fp0 = (vl_df['facepart'] == 0).sum()
    acne_present = acne_valid.shape[0]
    print(f"\nacne 어노테이션:")
    print(f"  facepart=0 전체 레코드: {total_fp0}")
    print(f"  여드름 있는 피험자: {acne_present}명 ({acne_present/total_fp0:.1%})")
    print(f"  여드름 없는 피험자: {total_fp0 - acne_present}명")
    print(f"  병변 수 분포: min={acne_counts.min()}, max={acne_counts.max()}, mean={acne_counts.mean():.1f}")
    
    # 병변 유형별 빈도
    from collections import Counter
    all_types = Counter()
    for lesions in acne_valid:
        if isinstance(lesions, list):
            for l in lesions:
                all_types[l.get('name', 'unknown')] += 1
    print(f"  병변 유형별 빈도: {dict(all_types)}")

In [ ]:
# 피험자 단위 어노테이션 집계 (이미지-부위별로 중복되므로 subject 단위로 집계)
# facepart=0(acne), facepart=8(chin_sagging) 등은 뷰(F)에서 하나씩
# → subject_id + facepart 기준 first() 로 집계
subject_anns = (
    vl_df[vl_df['angle'] == 0]  # F 뷰 (angle=0)
    .groupby(['subject_id', 'facepart'])[ann_cols]
    .first()
    .reset_index()
)

# 피험자별 라벨 테이블 (wide format)
per_subject = {}
for col in key_anns:
    sub_data = subject_anns[['subject_id', col]].dropna()
    for _, row in sub_data.iterrows():
        sid = row['subject_id']
        if sid not in per_subject:
            per_subject[sid] = {}
        per_subject[sid][col] = row[col]

subject_label_df = pd.DataFrame(per_subject).T.rename(columns=lambda c: c.replace('ann_', ''))
subject_label_df = subject_label_df.reset_index().rename(columns={'index': 'subject_id'})

print(f"피험자별 라벨 DataFrame: {subject_label_df.shape}")
subject_label_df.head(10)

In [ ]:
# 라벨 상관관계 히트맵
label_cols = [c for c in subject_label_df.columns if c != 'subject_id']
corr = subject_label_df[label_cols].astype(float).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('어노테이션 라벨 상관관계', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 장비별/각도별 분포

In [ ]:
device_map = {0: '디지털카메라', 1: '스마트패드', 2: '스마트폰'}
vl_df['device_name'] = vl_df['device'].map(device_map)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 장비별 피험자 수
dev_counts = vl_df.groupby('device_name')['subject_id'].nunique()
dev_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('장비별 피험자 수')
axes[0].set_xlabel('')

# 장비별 이미지 수
dev_img = vl_df.groupby('device_name')['filename'].nunique()
dev_img.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('장비별 고유 이미지 수')
axes[1].set_xlabel('')

# facepart별 레코드 수
fp_counts = vl_df['facepart'].value_counts().sort_index()
fp_counts.plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='black')
axes[2].set_title('facepart별 레코드 수')
axes[2].set_xlabel('facepart')

plt.tight_layout()
plt.show()

print("\n장비별 피험자 수:")
print(dev_counts)

## 7. 이미지-라벨 매칭 구조 확인 (가장 중요)

In [ ]:
# filename 패턴 분석
# JSON 안의 filename: '0001_01_F.jpg'
# JSON 파일명:        '0001_01_F_00.json'

sample = vl_df[['json_file', 'filename', 'subject_id', 'device', 'angle', 'facepart']].head(20)
print("파일명 패턴 (json → image):")
print(sample.to_string(index=False))

print("\n\n📌 이미지-라벨 매칭 규칙:")
print("  json 파일명: {subject_id}_{session}_{view}_{facepart:02d}.json")
print("  이미지 파일:  {subject_id}_{session}_{view}.jpg")
print("  → 이미지 1장에 facepart 0~8 라벨이 각각 별도 JSON으로 존재")
print("  → facepart의 bbox로 crop하여 부위별 학습 가능")

In [ ]:
# 피험자 1명의 이미지 수 확인
s001 = vl_df[vl_df['subject_id'] == '0001']
print(f"피험자 0001 총 레코드: {len(s001)}")
print(f"고유 이미지 파일: {s001['filename'].nunique()}장")
print(f"고유 JSON 파일:   {s001['json_file'].nunique()}개")
print(f"각도(angle) 종류: {sorted(s001['angle'].unique())}")
print(f"facepart 종류:    {sorted(s001['facepart'].unique())}")
print(f"device 종류:      {sorted(s001['device'].unique())}")

## 8. skin_type 코드 매핑 확인

In [ ]:
# JSON의 skin_type 정수 코드 vs meta_data.csv의 피부타입 이름 매핑
# subject_id는 JSON에서 '0001' 형식, meta_df는 1 형식이므로 int 변환
vl_df['subject_no'] = vl_df['subject_id'].astype(int)

# subject별 skin_type 추출 (중복 제거)
skin_code = vl_df.groupby('subject_no')['skin_type'].first().reset_index()
skin_mapped = skin_code.merge(meta_df[['subject_no', '얼굴피부타입']], on='subject_no', how='inner')

skin_type_map = (
    skin_mapped.groupby(['skin_type', '얼굴피부타입'])
    .size()
    .reset_index(name='count')
    .sort_values('skin_type')
)
print("skin_type 코드 → 피부타입 이름 매핑:")
print(skin_type_map.to_string(index=False))

## 9. 결측치 분석

In [ ]:
# 어노테이션 컬럼 결측률
null_rates = vl_df[ann_cols].isnull().mean().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
null_rates.plot(kind='barh', color='salmon', edgecolor='black')
plt.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='50% 기준선')
plt.xlabel('결측률')
plt.title('어노테이션 컬럼별 결측률')
plt.legend()
plt.tight_layout()
plt.show()

print("\n결측률 상세:")
null_df = pd.DataFrame({
    '결측률': null_rates,
    'non_null수': vl_df[ann_cols].notnull().sum()
})
print(null_df.to_string())

## 10. EDA 결론 및 학습 타겟 결정

### 10.1 데이터 규모 요약

| 구분 | JSON 수 | 피험자 수 |
|------|---------|---------|
| Validation | 12,519개 | 107명 |
| Training | 100,386개 | 858명 |
| **합계** | **~113,000개** | **~965명** |

| 항목 | 내용 |
|------|------|
| 장비 종류 | 3가지 (디지털카메라, 스마트패드, 스마트폰) |
| 촬영 각도 | 7가지 (F, Fb, Ft, L15, L30, R15, R30) |
| 얼굴 부위 | 9가지 (facepart 0~8) |
| skin_type | 0=건성, 1=지성, 3=복합건성, 4=복합지성, 5=중성 |

### 10.2 어노테이션 실제 스케일 (중요: 0-3이 아님)

| 속성 | facepart | 스케일 | 비고 |
|------|----------|--------|------|
| acne | 0 | OD형식 | 병변 좌표 리스트, null=여드름없음 |
| forehead_pigmentation | 1 | 0-5 | 6-class |
| forehead_wrinkle | 1 | **0-6** | **7-class, 베이스라인 추천** |
| glabellus_wrinkle | 2 | 0-6 | 7-class |
| l/r_perocular_wrinkle | 3,4 | 0-6 | 7-class |
| l/r_cheek_pore | 5,6 | 0-4 | 5-class |
| l/r_cheek_pigmentation | 5,6 | 0-5 | 6-class |
| lip_dryness | 7 | 0-4 | 5-class |
| chin_sagging | 8 | 0-5 | 6-class |

### 10.3 Phase 4 베이스라인 권장

**베이스라인 타겟**: `forehead_wrinkle` (facepart=1)
- 이유: 결측 없음, 직관적 시각 특징, 분포 비교적 균형

```python
# src/data/aihub_loader.py
BASELINE_TARGET = "forehead_wrinkle"  # 0-6, 7-class

# 사용 이미지: F뷰 + facepart=1 bbox crop
# 데이터 수: Training ~11,000장 (858명 × 3장비 × angle=0)
```

### 10.4 Multi-task 학습 최종 타겟 (7개, acne 별도 처리)

```python
MULTITASK_TARGETS = [
    "forehead_wrinkle",       # facepart 1  (0-6, 7-class)
    "forehead_pigmentation",  # facepart 1  (0-5, 6-class)
    "l_perocular_wrinkle",    # facepart 3  (0-6, 7-class)
    "l_cheek_pore",           # facepart 5  (0-4, 5-class)
    "l_cheek_pigmentation",   # facepart 5  (0-5, 6-class)
    "lip_dryness",            # facepart 7  (0-4, 5-class)
    "chin_sagging",           # facepart 8  (0-5, 6-class)
]
# acne는 별도 전처리 필요: 병변 count → 0-3 severity class 변환
```

### 10.5 person-level split 주의사항
- Training/Validation은 AI Hub에서 이미 분리됨 (피험자 단위 분리 보장)
- 같은 피험자가 train/val에 겹치지 않음 → leakage 없음 ✅

### 10.6 다음 단계 (Phase 3/4)
- [x] TL.zip 압축 해제 → Training 라벨 100,386개 확인
- [ ] VS.zip 압축 해제 → 실제 이미지-라벨 매칭 검증 (2GB)
- [ ] `src/data/dataset.py`: PyTorch Dataset 완성 (bbox crop + transforms)
- [ ] `src/models/cnn.py`: EfficientNet-B0 베이스라인 빌드
- [ ] Phase 4: 베이스라인 CNN 학습 (forehead_wrinkle, 7-class)

In [ ]:
# 최종 요약 출력
print("=" * 60)
print("EDA 완료 요약")
print("=" * 60)
print(f"\n[메타데이터]")
print(f"  총 피험자: {len(meta_df)}명")
print(f"  성별: {dict(meta_df['성별'].value_counts())}")
print(f"  연령대: {meta_df['나이'].min()}~{meta_df['나이'].max()}세")
print(f"  피부타입: {dict(meta_df['얼굴피부타입'].value_counts())}")

print(f"\n[Validation 라벨]")
print(f"  JSON 파일 수: {len(vl_df):,}개")
print(f"  피험자 수: {vl_df['subject_id'].nunique()}명")
print(f"  장비 수: {vl_df['device'].nunique()}종")

print(f"\n[학습 가능 어노테이션]")
for col in key_anns:
    if col in vl_df.columns:
        valid_count = vl_df[col].notna().sum()
        print(f"  {col.replace('ann_',''):35s}: {valid_count:5,}개")

print("\n✅ Phase 2 EDA 완료 → Phase 3 (TL 라벨 파싱 + Dataset 구현) 진행")